In [2]:
import pandas as pd
import numpy as np
import re

## Python Built-In String Methods

For most text munging tasks, Python's built-in string methods are sufficient.

### Splitting and Joining

- `str.split(sep)` → list of substrings split on `sep`.
- `str.strip()` → removes leading and trailing whitespace (including newlines).
- `sep.join(iterable)` → concatenates a sequence with `sep` as the delimiter.
  - Faster and more Pythonic than repeated `+` concatenation.

### Locating Substrings

| Method | Returns | On miss |
|--------|---------|--------|
| `in` keyword | `True` / `False` | — |
| `find(sub)` | index of first occurrence | `-1` |
| `rfind(sub)` | index of last occurrence | `-1` |
| `index(sub)` | index of first occurrence | raises `ValueError` |
| `count(sub)` | number of non-overlapping occurrences | `0` |

### Other Useful Methods

| Method | Effect |
|--------|-------|
| `replace(old, new)` | Replace all occurrences; pass `""` as `new` to delete |
| `lower()` / `upper()` | Case conversion |
| `startswith(prefix)` / `endswith(suffix)` | Prefix/suffix check |
| `strip()` / `lstrip()` / `rstrip()` | Remove whitespace from both/left/right |
| `ljust(w)` / `rjust(w)` | Pad to minimum width |

## Key Points

- Use `sep.join(list)` instead of `+` concatenation — it's faster and cleaner.
- `find` returns `-1` on miss; `index` raises `ValueError` — choose based on whether a miss is expected.
- `replace(old, "")` is a simple way to delete a substring.

In [3]:
val = "a,b, guido"

val.split(",")

['a', 'b', ' guido']

In [4]:
# split + strip to remove extra whitespace
pieces = [x.strip() for x in val.split(",")]

pieces

['a', 'b', 'guido']

In [5]:
"::".join(pieces)  # faster than + concatenation

'a::b::guido'

In [6]:
print("guido" in val)   # in keyword — simplest substring check
print(val.index(","))   # position of first comma
print(val.find(":"))    # -1 if not found
print(val.count(","))   # number of commas

True
1
-1
2


In [7]:
# val.index(":") would raise ValueError — substring not found
try:
    val.index(":")
except ValueError as e:
    print(e)

substring not found


In [8]:
print(val.replace(",", "::"))  # substitute
print(val.replace(",", ""))    # delete

a::b:: guido
ab guido


## Regular Expressions

The `re` module provides flexible pattern matching for complex string operations.

### Three Categories of `re` Functions

| Category | Functions |
|----------|-----------|
| Pattern matching | `search`, `match`, `findall`, `finditer` |
| Substitution | `sub`, `subn` |
| Splitting | `split` |

### `search` vs `match` vs `findall`

| Function | Behaviour |
|----------|-----------|
| `match(pattern, s)` | Match only at the **start** of the string; returns match object or `None` |
| `search(pattern, s)` | Match **anywhere** in the string; returns first match object or `None` |
| `findall(pattern, s)` | Returns **all** non-overlapping matches as a list |
| `finditer(pattern, s)` | Like `findall` but returns an iterator of match objects |

### Match Objects

- `.start()` / `.end()` — start and end index of the match.
- `.groups()` — tuple of captured groups (from parentheses in the pattern).

### Groups and Backreferences

- Wrap parts of the pattern in `()` to capture groups.
- In `sub`, use `\1`, `\2`, ... to reference captured groups in the replacement string.
- `findall` returns a **list of tuples** when the pattern has groups.

### Tips

- Use raw strings (`r"..."`) to avoid double-escaping backslashes.
- Use `re.compile(pattern, flags=...)` to create a reusable compiled regex — faster when applied to many strings.
- `re.IGNORECASE` makes the match case-insensitive.

## Key Points

- `match` anchors at the start; `search` finds the first match anywhere.
- `findall` with groups returns a list of tuples — one tuple per match.
- `sub(pattern, replacement, text)` supports `\1`, `\2` backreferences to groups.
- Compile regexes with `re.compile` when reusing the same pattern.

In [9]:
text = "foo bar\t baz \tqux"

re.split(r"\s+", text)  # split on one or more whitespace characters

['foo', 'bar', 'baz', 'qux']

In [10]:
# Compile for reuse
regex = re.compile(r"\s+")

print(regex.split(text))
print(regex.findall(text))  # all whitespace sequences matched

['foo', 'bar', 'baz', 'qux']
[' ', '\t ', ' \t']


In [11]:
text = """Dave dave@google.com
Steve steve@gmail.com
Rob rob@gmail.com
Ryan ryan@yahoo.com"""

pattern = r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,4}"
regex = re.compile(pattern, flags=re.IGNORECASE)

regex.findall(text)

['dave@google.com', 'steve@gmail.com', 'rob@gmail.com', 'ryan@yahoo.com']

In [12]:
# search: first match only, anywhere in the string
m = regex.search(text)

print(m)
print(text[m.start():m.end()])

<re.Match object; span=(5, 20), match='dave@google.com'>
dave@google.com


In [13]:
# match: anchored at start — returns None here since text begins with "Dave"
print(regex.match(text))

None


In [14]:
# sub: replace all matches
print(regex.sub("REDACTED", text))

Dave REDACTED
Steve REDACTED
Rob REDACTED
Ryan REDACTED


In [15]:
# Groups: () captures username, domain, suffix separately
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"
regex = re.compile(pattern, flags=re.IGNORECASE)

m = regex.match("wesm@bright.net")
m.groups()

('wesm', 'bright', 'net')

In [16]:
# findall with groups → list of tuples
regex.findall(text)

[('dave', 'google', 'com'),
 ('steve', 'gmail', 'com'),
 ('rob', 'gmail', 'com'),
 ('ryan', 'yahoo', 'com')]

In [17]:
# sub with backreferences \1, \2, \3 → reference captured groups
print(regex.sub(r"Username: \1, Domain: \2, Suffix: \3", text))

Dave Username: dave, Domain: google, Suffix: com
Steve Username: steve, Domain: gmail, Suffix: com
Rob Username: rob, Domain: gmail, Suffix: com
Ryan Username: ryan, Domain: yahoo, Suffix: com


## String Functions in pandas — `str` Accessor

Applying string operations to a Series with `.map()` fails on `NaN` values. pandas provides the `str` accessor on Series that:

- Applies operations **element-wise** across the whole array.
- **Propagates NA** values instead of raising an error.

Access via `series.str.<method>()`.

### Extension Type Advantage

Converting to `"string"` dtype before using `str` methods yields:
- `<NA>` instead of `NaN` for missing values.
- `boolean` dtype result instead of `object` dtype.

### Vectorized Indexing

- `series.str[n]` → slice or index into each string element.
- `series.str.get(n)` → equivalent to indexing into each element.
- `series.str[:n]` → slice the first `n` characters of each string.

### `str.extract`

`series.str.extract(pattern)` applies a regex with groups and returns a **DataFrame** — one column per captured group.

### Partial `str` Method Reference

| Method | Description |
|--------|-------------|
| `contains(pat)` | Boolean — whether each string contains the pattern |
| `findall(pat)` | List of all matches per element |
| `extract(pat)` | DataFrame of captured groups |
| `get(i)` | Retrieve i-th element |
| `split(sep)` | Split on separator |
| `join(sep)` | Join strings in each element with separator |
| `replace(pat, rep)` | Replace pattern with replacement |
| `strip` / `lstrip` / `rstrip` | Trim whitespace |
| `lower` / `upper` | Case conversion |
| `len` | Length of each string |
| `slice(start, end)` | Slice each string |
| `startswith` / `endswith` | Prefix/suffix check |
| `count(pat)` | Count occurrences of pattern |
| `pad` / `center` | Add whitespace padding |
| `repeat(n)` | Repeat each string `n` times |
| `cat(sep)` | Concatenate strings element-wise |
| `match(pat)` | True/False whether element matches pattern |
| `get_dummies(sep)` | One-hot-encode delimited multi-category strings |

## Key Points

- `series.str` accessor skips `NaN` and propagates it in the result — use it instead of `.map(func)` for string operations.
- `series.astype("string")` first gives `<NA>` and `boolean` typed results from `str` methods.
- `str.findall` + `str[0]` extracts the first match per element.
- `str.extract(pattern)` returns a DataFrame with one column per captured group.

In [18]:
data = pd.Series({
    "Dave": "dave@google.com",
    "Steve": "steve@gmail.com",
    "Rob": "rob@gmail.com",
    "Wes": np.nan
})

data

Dave     dave@google.com
Steve    steve@gmail.com
Rob        rob@gmail.com
Wes                  NaN
dtype: object

In [19]:
data.isna()

Dave     False
Steve    False
Rob      False
Wes       True
dtype: bool

In [20]:
# str.contains propagates NaN — result is object dtype
data.str.contains("gmail")

Dave     False
Steve     True
Rob       True
Wes        NaN
dtype: object

In [21]:
# With string extension type: NaN → <NA>, result is boolean dtype
data_ext = data.astype("string")

data_ext.str.contains("gmail")

Dave     False
Steve     True
Rob       True
Wes       <NA>
dtype: boolean

In [22]:
# Regex with str.findall
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"

data.str.findall(pattern, flags=re.IGNORECASE)

Dave     [(dave, google, com)]
Steve    [(steve, gmail, com)]
Rob        [(rob, gmail, com)]
Wes                        NaN
dtype: object

In [23]:
# Vectorized indexing: get the first match tuple, then retrieve the domain
matches = data.str.findall(pattern, flags=re.IGNORECASE).str[0]

print(matches)
print()
print(matches.str.get(1))  # domain name

Dave     (dave, google, com)
Steve    (steve, gmail, com)
Rob        (rob, gmail, com)
Wes                      NaN
dtype: object

Dave     google
Steve     gmail
Rob       gmail
Wes         NaN
dtype: object


In [24]:
# Slice first 5 characters of each element
data.str[:5]

Dave     dave@
Steve    steve
Rob      rob@g
Wes        NaN
dtype: object

In [25]:
# str.extract: returns a DataFrame — one column per captured group
data.str.extract(pattern, flags=re.IGNORECASE)

,0,1,2
Dave,dave,google,com
Steve,steve,gmail,com
Rob,rob,gmail,com
Wes,NaN,NaN,NaN
